# Chapter 20: Building Production AI Systems
## Notebook 01 — Reliability Patterns

Real services face bursts, flaky dependencies, and overload. Three patterns handle most of it: **rate limiting**, **circuit breaking**, and **backoff**.

### What you'll learn

| Topic | Section |
|-------|--------|
| Token-bucket rate limiting | §1 |
| Circuit breaker state machine | §2 |
| Exponential backoff with jitter | §3 |

**Time estimate:** 3.5 hours

### Key concepts

- **Token bucket** — allows short bursts up to a capacity, then sustains a steady rate.
- **Circuit breaker** — stop hammering a failing dependency; fail fast, recover gradually.
- **Backoff + jitter** — spread retries to avoid synchronized thundering herds.

---
*Generated by Berta AI*

## 1. Rate limiting

The token bucket grants requests while tokens remain, refilling at a fixed rate — tolerating bursts without allowing sustained overload.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from reliability import TokenBucket

tb = TokenBucket(rate=2.0, capacity=3.0)  # 2/sec, burst 3
results = [tb.allow(now=0.0) for _ in range(5)]   # burst at t=0
print("burst of 5 at t=0:", results)              # 3 allowed, then blocked
print("after 1s:", tb.allow(now=1.0))             # ~2 refilled

## 2. Circuit breaker

After repeated failures the breaker trips **open** and fails fast, sparing the dependency, then probes recovery via **half-open**.

In [ ]:
from reliability import CircuitBreaker, CircuitOpen

cb = CircuitBreaker(fail_threshold=2, reset_timeout=5.0)
def flaky():
    raise ValueError("boom")
for t in range(3):
    try:
        cb.call(flaky, now=float(t))
    except (ValueError, CircuitOpen) as e:
        print(f"t={t}: {type(e).__name__}, state={cb.state}")

---

## Summary

Token buckets tame bursts, circuit breakers contain failing dependencies, and backoff with jitter prevents retry storms. Together they keep a service standing under stress.

---
*Generated by Berta AI | Berta Chapters*